# Mood-based Movie Recommendations — TF-IDF Evaluation

**Replaces `scripts/run_mood_queries.py`.** This notebook loads the team's **existing TF-IDF baseline**, executes the 28 shared mood queries, shows and exports ranked movie results, creates a **human-labeling sheet**, and computes evaluation metrics **only when sufficient ratings exist**.

**Evaluation question:** *How well does lexical TF-IDF retrieval match a user's desired movie mood?*

### Input files (paths relative to the repository root)

- `src/baseline.py` and `src/tfidf_text.py` — the previously created baseline + consistent query normalization
- `evaluation/mood_queries.json` — 28 fixed test queries
- `models/tfidf_wordnet/` — preferred trained index (or set the correct index below)
- Optional: `data/processed/sample_clean_movies.csv` — plot snippets to help humans assess recommendations

**Outputs** are saved to `evaluation/results/` and `evaluation/mood_judgments.csv`. Make your edits to the judgments CSV **outside** this notebook, then rerun the metrics cells. The notebook preserves existing judgments when it adds new recommendations.

> **Important:** Similarity scores are **not** relevance labels. Precision@5, MRR@5 and nDCG@5 need human judgments (0 = irrelevant, 1 = partially relevant, 2 = highly relevant). Do not report model accuracy before obtaining labels. For an unjudged 100-movie sample, report only retrieval diagnostics.

## 1. Set up paths and imports

In PyCharm, open this notebook inside `notebooks/` and select the same Python interpreter used for `src/baseline.py`.

Install packages once in the terminal: `python -m pip install -r requirements-baseline.txt` and `python -m nltk.downloader wordnet omw-1.4`.

This notebook will locate the repository root automatically when run from the project root or its `notebooks/` folder.

In [4]:
%pip install spacy nltk

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached confection-1.3.4-py3-none-any.whl.metadata (19 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached cloudpathlib-0.26.0-py3-none-any.whl.metadata (16 kB)
  Using cached smart_open-9.0.4-py3-none-any.whl.metadata (25 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 24.7 MB/s  0:00:00 eta 0:00:01
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
Using cached confection-1.3.4-py3-none-any.whl (35 kB)
Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl (29 kB)
Using cached

In [1]:
from pathlib import Path
import json
import sys
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display


# Find your repository root. No hardcoded user-specific paths.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in candidates if (p / "src" / "baseline.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not locate src/baseline.py. Open this notebook from the "
        "Move-recommendation-system repository (or its notebooks/ folder)."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.baseline import (BaselineError, TfidfMovieRecommender, _normalize_query)

QUERY_FILE = ROOT / "evaluation" / "mood_queries.json"
OUTPUT_DIR = ROOT / "evaluation" / "results"
JUDGMENTS_FILE = ROOT / "evaluation" / "tfidf_mood_judgments.csv"
REFERENCE_DATA = ROOT / "data" / "processed" / "sample_clean_movies.csv"
INDEX_DIR = ROOT / "models" / "tfidf_sample"   # Change if yours is elsewhere.
TOP_K = 5

if not INDEX_DIR.is_dir():
    alternative = ROOT / "models" / "tfidf_sample"
    if alternative.is_dir():
        INDEX_DIR = alternative
        print("Preferred wordnet index not found; using:", INDEX_DIR)

if TOP_K < 1:
    raise ValueError("TOP_K must be positive.")
print("Repository:", ROOT)
print("Index:", INDEX_DIR)
print("Query file:", QUERY_FILE)

Repository: /Users/rh/Documents/GitHub/Move-recommendation-system
Index: /Users/rh/Documents/GitHub/Move-recommendation-system/models/tfidf_sample
Query file: /Users/rh/Documents/GitHub/Move-recommendation-system/evaluation/mood_queries.json


## 2. Load the trained baseline and test queries

The model's saved configuration controls how queries are cleaned. For Vojta's already-lemmatized `text_tfidf` document column, use a model trained with `query_mode='wordnet'`. **Do not refit the TF-IDF vectorizer separately for each query.**

If the index is missing, first train it from your project root:

```bash
python -m src.baseline train --data data/processed/sample_clean_movies.csv --output models/tfidf_wordnet --text-column text_tfidf --query-mode wordnet
```

In [2]:
if not QUERY_FILE.is_file():
    raise FileNotFoundError(f"No query file: {QUERY_FILE}. Place mood_queries.json under evaluation/.")
if not INDEX_DIR.is_dir():
    raise FileNotFoundError(
        f"No saved model in {INDEX_DIR}. Train it using the command in the previous cell."
    )
model = TfidfMovieRecommender.load(INDEX_DIR)  # Load only a trusted index made by your team.
queries = json.loads(QUERY_FILE.read_text(encoding="utf-8"))
if not isinstance(queries, list) or not queries:
    raise ValueError("mood_queries.json must contain a nonempty list.")
if any(not isinstance(x, dict) or not isinstance(x.get("query_id"), str)
       or not isinstance(x.get("query"), str) for x in queries):
    raise ValueError("Every item needs string fields query_id and query.")
query_ids = [x["query_id"] for x in queries]
if len(query_ids) != len(set(query_ids)):
    raise ValueError("query_id values must be unique.")

print(f"Loaded {len(queries)} queries; index has {len(model.metadata):,} movies")
print("Text column:", model.config.text_column)
print("Saved query mode:", model.config.query_mode)
print("Index statistics:", model.stats)
if model.config.query_mode != "wordnet":
    print("WARNING: For consistent preprocessing with Vojta's text_tfidf, train an index with --query-mode wordnet.")
display(pd.DataFrame(queries)[["query_id", "category", "query", "test_purpose"]].head(10))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loaded 28 queries; index has 100 movies
Text column: text_tfidf
Saved query mode: wordnet
Index statistics: {'input_rows': 100, 'indexed_rows': 100, 'dropped_invalid_rows': 0, 'truncated_documents': 7, 'n_features': 50000, 'nonzero_elements': 151241, 'text_column': 'text_tfidf'}


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,query_id,category,query,test_purpose
0,M01,direct_mood,I feel like watching something gloomy and thri...,dark suspense
1,M02,direct_mood,"I want a cozy, comforting movie for a rainy ev...",cozy comfort
2,M03,direct_mood,Something eerie and unsettling that slowly bui...,slow-burn dread
3,M04,direct_mood,An uplifting and hopeful film that makes me fe...,hopeful uplifting
4,M05,direct_mood,I am in the mood for something melancholic and...,melancholy introspection
5,M06,direct_mood,"Give me a funny, chaotic movie that does not t...",chaotic comedy
6,M07,mood_theme,A tense mystery set in an isolated place with ...,tension isolation mystery
7,M08,mood_theme,A bittersweet coming-of-age story about friend...,bittersweet coming of age
8,M09,mood_theme,A warm romance about two people finding each o...,warm romance
9,M10,mood_theme,"A haunting movie about grief, memory and loss.",grief memory loss


## 3. Run all mood queries through TF-IDF

Record the **original query**, the **preprocessed query**, the ranked movies, and an explicit status. Empty or out-of-vocabulary queries should not stop the batch.

In [5]:
query_results = []
for item in queries:
    q = item["query"]
    rec = {
        "query_id": item["query_id"],
        "category": item.get("category", ""),
        "query": q,
        "test_purpose": item.get("test_purpose", ""),
        "method": "tfidf",
        "text_source": model.config.text_column,
        "query_mode": model.config.query_mode,
        "processed_query": "",
        "recommendations": [],
    }
    try:
        rec["processed_query"] = _normalize_query(q, model.config.query_mode)
        rec["recommendations"] = model.search(q, top_k=TOP_K)
        rec["status"] = "ok" if rec["recommendations"] else "no_matches"
    except (BaselineError, RuntimeError, ValueError, TypeError) as exc:
        rec["status"] = "error"
        rec["error"] = str(exc)
    query_results.append(rec)

summary = pd.DataFrame([
    {"query_id": r["query_id"], "category": r["category"],
     "original_query": r["query"], "processed_query": r["processed_query"],
     "status": r["status"], "returned": len(r["recommendations"]),
     "error": r.get("error", "")}
    for r in query_results
])
print("Status counts:", dict(Counter(summary["status"])))
display(summary)
if (summary["status"] == "error").all():
    print("All queries errored. Check saved query_mode and WordNet corpus: python -m nltk.downloader wordnet omw-1.4")

The query has no usable words after normalization.
No words in the query appear in the fitted vocabulary.


Status counts: {'ok': 25, 'no_matches': 2, 'error': 1}


,query_id,category,original_query,processed_query,status,returned,error
0,M01,direct_mood,I feel like watching something gloomy and thri...,gloomy thrilling,ok,3,
1,M02,direct_mood,"I want a cozy, comforting movie for a rainy ev...",cozy comforting movie rainy evening,ok,5,
2,M03,direct_mood,Something eerie and unsettling that slowly bui...,eerie unsettling slowly build tension,ok,5,
3,M04,direct_mood,An uplifting and hopeful film that makes me fe...,uplifting hopeful film feel better,ok,5,
4,M05,direct_mood,I am in the mood for something melancholic and...,melancholic reflective,ok,1,
5,M06,direct_mood,"Give me a funny, chaotic movie that does not t...",funny chaotic movie seriously,ok,5,
6,M07,mood_theme,A tense mystery set in an isolated place with ...,tense mystery set isolated place secret,ok,5,
7,M08,mood_theme,A bittersweet coming-of-age story about friend...,bittersweet coming age story friendship growin...,ok,5,
8,M09,mood_theme,A warm romance about two people finding each o...,warm romance people finding unexpectedly,ok,5,
9,M10,mood_theme,"A haunting movie about grief, memory and loss.",haunting movie grief memory loss,ok,5,


## 4. Inspect top recommendations

This table is **retrieval output**, not evaluation. Similarity measures lexical match only; high score does not prove the mood fits.

In [6]:
flat_rows = []
for r in query_results:
    for movie in r["recommendations"]:
        flat_rows.append({
            "query_id": r["query_id"], "category": r["category"],
            "query": r["query"], "processed_query": r["processed_query"],
            "rank": movie["rank"], "movie_id": str(movie["movie_id"]),
            "title": movie["title"], "score": movie["score"],
            "genres": movie.get("genres", ""),
            "release_year": movie.get("release_year", ""),
        })
rank_columns = ["query_id", "category", "query", "processed_query", "rank", "movie_id", "title", "score", "genres", "release_year"]
rankings = pd.DataFrame(flat_rows, columns=rank_columns)
if rankings.empty:
    print("No recommendations found. Review the query statuses above.")
else:
    display(rankings.head(20))
    display(rankings.groupby("query_id", as_index=False).head(1)[["query_id", "query", "title", "score"]].head(15))

,query_id,category,query,processed_query,rank,movie_id,title,score,genres,release_year
0,M01,direct_mood,I feel like watching something gloomy and thri...,gloomy thrilling,1,tt0032961,Quiet Wedding,0.027949,Comedy|Romance,1941
1,M01,direct_mood,I feel like watching something gloomy and thri...,gloomy thrilling,2,tt2011351,Chips,0.026642,Comedy|Drama,2012
2,M01,direct_mood,I feel like watching something gloomy and thri...,gloomy thrilling,3,tt0082162,Chashme Buddoor,0.016460,Comedy|Romance,1981
3,M02,direct_mood,"I want a cozy, comforting movie for a rainy ev...",cozy comforting movie rainy evening,1,tt0075342,The Town That Dreaded Sundown,0.022548,Horror|Mystery|Thriller|Crime,1976
4,M02,direct_mood,"I want a cozy, comforting movie for a rainy ev...",cozy comforting movie rainy evening,2,tt0277106,Le Loup blanc,0.016129,TV Movie,1977
5,M02,direct_mood,"I want a cozy, comforting movie for a rainy ev...",cozy comforting movie rainy evening,3,tt0124008,Miss Morison's Ghosts,0.015350,Horror|Thriller|TV Movie,1981
6,M02,direct_mood,"I want a cozy, comforting movie for a rainy ev...",cozy comforting movie rainy evening,4,tt0024034,42nd Street,0.013614,Comedy|Drama|Romance,1933
7,M02,direct_mood,"I want a cozy, comforting movie for a rainy ev...",cozy comforting movie rainy evening,5,tt0104695,Leap of Faith,0.013420,Comedy|Drama|Romance,1992
8,M03,direct_mood,Something eerie and unsettling that slowly bui...,eerie unsettling slowly build tension,1,tt2180016,Iceberg Slim: Portrait of a Pimp,0.021311,Documentary,2012
9,M03,direct_mood,Something eerie and unsettling that slowly bui...,eerie unsettling slowly build tension,2,tt0057280,Crypt of the Vampire,0.019302,Horror,1964


,query_id,query,title,score
0,M01,I feel like watching something gloomy and thri...,Quiet Wedding,0.027949
3,M02,"I want a cozy, comforting movie for a rainy ev...",The Town That Dreaded Sundown,0.022548
8,M03,Something eerie and unsettling that slowly bui...,Iceberg Slim: Portrait of a Pimp,0.021311
13,M04,An uplifting and hopeful film that makes me fe...,Anne of the Thousand Days,0.029445
18,M05,I am in the mood for something melancholic and...,Boogie Woogie,0.020725
19,M06,"Give me a funny, chaotic movie that does not t...",All We Ever Wanted,0.033765
24,M07,A tense mystery set in an isolated place with ...,Hum Kaun Hai,0.029075
29,M08,A bittersweet coming-of-age story about friend...,Girlfriend Boyfriend,0.048218
34,M09,A warm romance about two people finding each o...,Dangerous Love Affairs,0.030394
39,M10,"A haunting movie about grief, memory and loss.",Hôtel des Invalides,0.025037


## 5. Export machine-readable results

The JSON keeps complete per-query results and statuses. The CSV is convenient for comparing TF-IDF, semantic search and hybrid retrieval later.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
json_path = OUTPUT_DIR / "tfidf_mood_results.json"
csv_path = OUTPUT_DIR / "tfidf_mood_rankings.csv"
status_path = OUTPUT_DIR / "tfidf_query_status.csv"
json_path.write_text(json.dumps(query_results, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
rankings.to_csv(csv_path, index=False, encoding="utf-8-sig")
summary.to_csv(status_path, index=False, encoding="utf-8-sig")
print("Saved:", json_path.relative_to(ROOT))
print("Saved:", csv_path.relative_to(ROOT))
print("Saved:", status_path.relative_to(ROOT))

Saved: evaluation/results/tfidf_mood_results.json
Saved: evaluation/results/tfidf_mood_rankings.csv
Saved: evaluation/results/tfidf_query_status.csv


## 6. Create a human relevance-labeling sheet

**A human must judge** how well each movie actually fits the mood query, using movie context (the plot, a trailer or other reliable movie information), not only the TF-IDF score.

Fill **`relevance`** with:

- **0** — does not fit the requested mood
- **1** — somewhat fits / partly relevant
- **2** — strongly fits the requested mood

Add optional reviewer name and notes. **Do not score based solely on the title or similarity score.**

This cell creates `evaluation/mood_judgments.csv` if missing. On later runs, it **preserves existing labels and appends only new query/movie pairs**, so a rerun won't erase your work. The same query/movie judgments can be reused to evaluate semantic and hybrid outputs, though you'll need to judge their additional retrieved movies too.

In [8]:
# Optional plot snippet from the sample makes manual annotation easier.
plot_lookup = {}
if REFERENCE_DATA.is_file():
    try:
        ref = pd.read_csv(REFERENCE_DATA, sep=";", encoding="utf-8-sig",
                          usecols=lambda c: c in {"imdb_id", "clean_plot"},
                          dtype={"imdb_id": "string"})
        if {"imdb_id", "clean_plot"}.issubset(ref.columns):
            ref = ref.drop_duplicates(subset=["imdb_id"])
            plot_lookup = dict(zip(ref["imdb_id"].astype(str), ref["clean_plot"].fillna("").astype(str)))
    except (OSError, ValueError) as exc:
        print("Could not load optional plot snippets:", exc)

label_columns = ["query_id", "movie_id", "query", "title", "genres", "plot_snippet", "relevance", "reviewer", "notes"]
new_labels = pd.DataFrame([
    {"query_id": row.query_id, "movie_id": str(row.movie_id),
     "query": row.query, "title": row.title,
     "genres": row.genres,
     "plot_snippet": plot_lookup.get(str(row.movie_id), "")[:500],
     "relevance": "", "reviewer": "", "notes": ""}
    for row in rankings.itertuples(index=False)
], columns=label_columns)
new_labels = new_labels.drop_duplicates(subset=["query_id", "movie_id"])

JUDGMENTS_FILE.parent.mkdir(parents=True, exist_ok=True)
if JUDGMENTS_FILE.is_file():
    old_labels = pd.read_csv(JUDGMENTS_FILE, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    required_labels = {"query_id", "movie_id", "relevance"}
    if not required_labels.issubset(old_labels.columns):
        raise ValueError(f"Existing {JUDGMENTS_FILE} is missing {required_labels - set(old_labels.columns)}. Back it up and fix columns before rerunning.")
    if old_labels.duplicated(["query_id", "movie_id"]).any():
        raise ValueError("Duplicate query_id/movie_id pairs in judgments CSV. Fix these manually.")
    for col in label_columns:
        if col not in old_labels:
            old_labels[col] = ""
    keys = set(zip(old_labels.query_id, old_labels.movie_id))
    to_append = new_labels.loc[[
        (q, mid) not in keys for q, mid in zip(new_labels.query_id, new_labels.movie_id)
    ]]
    judgments = pd.concat([old_labels[label_columns], to_append], ignore_index=True)
    if not to_append.empty:
        judgments.to_csv(JUDGMENTS_FILE, index=False, encoding="utf-8-sig")
    print(f"Preserved {len(old_labels)} existing judgments; added {len(to_append)} new pairs")
else:
    judgments = new_labels.copy()
    judgments.to_csv(JUDGMENTS_FILE, index=False, encoding="utf-8-sig")
    print(f"Created rating sheet with {len(judgments)} unique query/movie pairs")
print("Rate the movies in:", JUDGMENTS_FILE.relative_to(ROOT))
display(judgments.head(10))

Created rating sheet with 119 unique query/movie pairs
Rate the movies in: evaluation/mood_judgments.csv


,query_id,movie_id,query,title,genres,plot_snippet,relevance,reviewer,notes
0,M01,tt0032961,I feel like watching something gloomy and thri...,Quiet Wedding,Comedy|Romance,"A young couple become engaged, but enjoy a num...",,,
1,M01,tt2011351,I feel like watching something gloomy and thri...,Chips,Comedy|Drama,Tadashi Imamura is a burglar who idolises Ozak...,,,
2,M01,tt0082162,I feel like watching something gloomy and thri...,Chashme Buddoor,Comedy|Romance,Womanizing slackers Omi and Jai attempt to woo...,,,
3,M02,tt0075342,"I want a cozy, comforting movie for a rainy ev...",The Town That Dreaded Sundown,Horror|Mystery|Thriller|Crime,When two young lovers are savagely beaten and ...,,,
4,M02,tt0277106,"I want a cozy, comforting movie for a rainy ev...",Le Loup blanc,TV Movie,"In the 18th century, the peasants of the fores...",,,
5,M02,tt0124008,"I want a cozy, comforting movie for a rainy ev...",Miss Morison's Ghosts,Horror|Thriller|TV Movie,Two British women claim to have been thrown in...,,,
6,M02,tt0024034,"I want a cozy, comforting movie for a rainy ev...",42nd Street,Comedy|Drama|Romance,Broadway director Julian Marsh needs just one ...,,,
7,M02,tt0104695,"I want a cozy, comforting movie for a rainy ev...",Leap of Faith,Comedy|Drama|Romance,Jonas Nightengale is a fraudulent Christian fa...,,,
8,M03,tt2180016,Something eerie and unsettling that slowly bui...,Iceberg Slim: Portrait of a Pimp,Documentary,"'Iceberg Slim: Portrait of a Pimp', examines t...",,,
9,M03,tt0057280,Something eerie and unsettling that slowly bui...,Crypt of the Vampire,Horror,Count Karnstein sends for a doctor to help his...,,,


## 7. Calculate evaluation metrics from completed human judgments

After filling `evaluation/mood_judgments.csv` and saving it, **rerun this cell and those below**. An unjudged or partially judged query is **not counted as evaluated**.

- **Precision@K**: share of the top K that is at least partly relevant (grade 1 or 2)
- **MRR@K**: reciprocal rank of the first at-least-partly-relevant recommendation
- **nDCG@K**: graded relevance ranking quality (0/1/2)

nDCG here uses ideal ordering of the **K judged retrieved movies** (a *judged-pool* nDCG), not a complete set of all relevant movies in the corpus. **Recall@K is not calculated** because total relevant items in the full corpus are unknown.

A query with fewer than K results is excluded from these three aggregate metrics; retrieval coverage is reported separately.

In [ ]:
from math import log2

def metrics_at_k(grades, k=5):
    """Grades ordered by rank; all k predictions must have grades in {0,1,2}."""
    vals = np.asarray(grades, dtype=int)
    if len(vals) != k or not np.isin(vals, [0, 1, 2]).all():
        raise ValueError("Need k valid human relevance grades (0, 1 or 2).")
    binary = vals >= 1
    precision = float(binary.mean())
    relevant_positions = np.flatnonzero(binary)
    mrr = float(1 / (relevant_positions[0] + 1)) if len(relevant_positions) else 0.0
    discounts = np.log2(np.arange(2, k + 2))
    dcg = float(np.sum((2.0 ** vals - 1) / discounts))
    ideal = np.sort(vals)[::-1]
    idcg = float(np.sum((2.0 ** ideal - 1) / discounts))
    ndcg = dcg / idcg if idcg else 0.0
    return {f"precision@{k}": precision, f"mrr@{k}": mrr, f"ndcg@{k}_judged_pool": ndcg}

# Reload the CSV from disk so changes in Excel / PyCharm are picked up.
labels = pd.read_csv(JUDGMENTS_FILE, dtype=str, keep_default_na=False, encoding="utf-8-sig")
if not {"query_id", "movie_id", "relevance"}.issubset(labels.columns):
    raise ValueError("Judgments CSV needs query_id, movie_id, relevance columns")
if labels.duplicated(["query_id", "movie_id"]).any():
    raise ValueError("Duplicate query_id / movie_id judgments")
labels["relevance"] = labels["relevance"].str.strip()
invalid = labels.loc[(labels["relevance"] != "") & ~labels["relevance"].isin(["0", "1", "2"])]
if not invalid.empty:
    print("Warning: Invalid ratings found; these entries will be treated as unjudged:")
    display(invalid[["query_id", "movie_id", "relevance"]].head(10))
labels["grade"] = pd.to_numeric(labels["relevance"].where(labels["relevance"].isin(["0", "1", "2"])), errors="coerce")

scored = rankings.merge(labels[["query_id", "movie_id", "grade"]],
                        on=["query_id", "movie_id"], how="left", validate="many_to_one")
metric_rows = []
for qid, group in scored.groupby("query_id", sort=False):
    group = group.sort_values("rank")
    if len(group) != TOP_K or group["grade"].isna().any():
        continue
    row = {"query_id": qid, "category": group["category"].iloc[0],
           "query": group["query"].iloc[0]}
    row.update(metrics_at_k(group["grade"].astype(int).tolist(), TOP_K))
    metric_rows.append(row)
metrics_df = pd.DataFrame(metric_rows)
print(f"Fully judged queries with {TOP_K} results: {len(metrics_df)} / {len(queries)}")
print(f"Queries with at least one retrieval: {(summary.returned > 0).sum()} / {len(summary)}")
print(f"Average returned movies per query: {summary.returned.mean():.2f} / {TOP_K}")
if metrics_df.empty:
    print("No validated relevance metrics yet. Fill the blank 'relevance' column in evaluation/mood_judgments.csv, save, then rerun this section.")
else:
    metric_cols = [f"precision@{TOP_K}", f"mrr@{TOP_K}", f"ndcg@{TOP_K}_judged_pool"]
    display(pd.DataFrame([metrics_df[metric_cols].mean().to_dict()], index=["Mean across fully judged queries"]).round(3))
    display(metrics_df.round(3))
    display(metrics_df.groupby("category")[metric_cols].agg(["mean", "count"]).round(3))

## 8. Visualize evaluated quality (when labeled)

Each bar is the **mean over fully judged queries only**. Do not extrapolate quality from the 100-movie smoke-test sample to the full movie corpus.

In [ ]:
if not metrics_df.empty:
    import matplotlib.pyplot as plt
    metric_cols = [f"precision@{TOP_K}", f"mrr@{TOP_K}", f"ndcg@{TOP_K}_judged_pool"]
    ax = metrics_df[metric_cols].mean().plot(kind="bar", rot=15, figsize=(8, 4), ylim=(0, 1))
    ax.set_ylabel("Mean score")
    ax.set_title(f"TF-IDF mood recommendation evaluation (n={len(metrics_df)} judged queries)")
    plt.tight_layout()
    plt.show()
else:
    print("Chart will appear after human relevance labels have been completed.")

## 9. Identify failure cases and save evaluation evidence

Automatic failure flags include empty/missing matches, processing errors, and (once labeled) queries with zero relevant items in the top K. **These are prompts for manual error analysis, not proof of why a query failed.**

Common issues to examine: vocabulary mismatch, long-subtitle dominance, out-of-vocabulary words, overly conversational query wording, **negation** (e.g., *not scary*), and absence of relevant movies from the sample.

In [ ]:
flags = summary.loc[summary["status"].isin(["error", "no_matches"])].copy()
if not metrics_df.empty:
    zero_hits = metrics_df.loc[metrics_df[f"precision@{TOP_K}"] == 0, ["query_id"]].copy()
    if not zero_hits.empty:
        extra = summary.merge(zero_hits, on="query_id", how="inner")
        extra["status"] = "no_human_relevant_results_in_top_k"
        flags = pd.concat([flags, extra], ignore_index=True)
print("Queries flagged for error analysis:", len(flags))
display(flags[["query_id", "original_query", "processed_query", "status", "returned", "error"]])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
flags.to_csv(OUTPUT_DIR / "tfidf_failure_candidates.csv", index=False, encoding="utf-8-sig")
if not metrics_df.empty:
    metrics_df.to_csv(OUTPUT_DIR / "tfidf_labeled_metrics.csv", index=False, encoding="utf-8-sig")
    print("Saved labeled evaluation metrics.")
print("Saved failure candidates for inspection.")

## 10. Notes for your report and hybrid comparison

- **Model**: TF-IDF fitted on `text_tfidf`, queried with the saved normalization mode, cosine similarity through normalized sparse vectors.
- **Dataset**: First evaluate functionality on 100 sampled movies, then rerun on a representative/full corpus.
- **Same conditions**: Use the same query list, same movie candidate pool and same human relevance judgments for TF-IDF, semantic retrieval and any hybrid model.
- **Not accuracy**: Retrieval coverage and similarity scores do not measure mood relevance.
- **No labels, no Precision / MRR / nDCG**: Obtain human assessments first; use an independent judging procedure as much as possible.
- **Remember negative preferences**: Vojta's current preprocessing drops `not`; a negation-related failure needs to be acknowledged in the failure journal.
- **Optional document comparison**: To test plot-only vs. subtitle-only later, train *separate* indexes on separate columns after the data handoff contains both columns. This notebook evaluates whichever index `INDEX_DIR` points to.
- **Generated outputs**: Results and judgment files can be large; add only appropriate and permitted files to GitHub. Keep restricted subtitle content out of public commits.